# 05 · Quality gates, CI/CD and production monitoring
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/ragas-evals-enterprise/blob/main/notebooks/05_quality_gates_and_ci.ipynb)

**Goal:** turn evaluation from something you *run* into something that **protects** production:
written release criteria, a CLI that exits non-zero when they fail, a CI workflow, experiment tracking with Ragas'
own API, and reference-free monitoring of live traffic.

| | |
|---|---|
| ⏱️ Time | ~25 minutes |
| 💵 API cost | ~$0.05 |

### 🔑 Before you run anything
* **Colab:** click the 🔑 **Secrets** icon in the left sidebar → *Add new secret* → name `OPENAI_API_KEY`,
  paste your key, and switch on **Notebook access**. (If you skip this, the setup cell asks you to paste the key.)
* **Local:** copy `.env.example` to `.env` and put your key there. See the README for the full setup.

The setup cell clones the project, installs the pinned requirements (~1 min on Colab), and loads your key.
If Colab prints pip *dependency-resolver warnings* about other preinstalled packages, they are harmless.

In [1]:
# ⚙️ SETUP: run this cell first. It works in Google Colab and on your laptop.
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/baluragala/ragas-evals-enterprise.git"  # ← change this if you forked the repo

if "google.colab" in sys.modules:                       # running in Colab: fetch the project + dependencies
    if not Path("/content/ragas-evals-enterprise").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ragas-evals-enterprise"], check=True)
    os.chdir("/content/ragas-evals-enterprise")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Find the project root (works from notebooks/ or the repo root) and make `aurora_evals` importable.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "aurora_evals").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["RAGAS_DO_NOT_TRACK"] = "true"   # turn off ragas telemetry (it also makes evals ~5x faster; see docs)

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)

from aurora_evals.config import setup_environment
setup_environment()   # finds OPENAI_API_KEY in your env / .env / Colab Secrets, or asks you to paste it
print("✅ Ready. Project root:", ROOT)

✅ Ready. Project root: /Users/balakrishna/Training/gen-ai/ragas-evals-enterprise


## 1. Release criteria as code: `config/quality_gates.yaml`

Thresholds are **product decisions** agreed with the business owner, versioned next to the code. Three gate types:

* **overall**: dataset mean ≥ min
* **slice**: the same check on a subset (e.g. only adversarial questions); catches what averages hide
* **regression**: may not drop more than `max_drop` below the **reference baseline**

In [2]:
print((ROOT / "config" / "quality_gates.yaml").read_text())

# ------------------------------------------------------------------------------------------
# Quality gates: the release criteria for the Aurora assistant.
#
# A change (new prompt, new chunking, new model) may ship only if ALL gates pass.
# Thresholds are product decisions, agreed with the business owner, not engineering defaults:
#   - "hard" metrics (compliance) get strict thresholds, even 1.0;
#   - "soft" metrics (tone) get looser ones;
#   - every number here should be revisited when the golden dataset changes.
#
# Gate types
#   overall     : mean of the metric over the whole dataset must be >= min
#   slices      : same, but only over rows where <column> == <value>. This catches failures
#                 that an average hides (e.g. 2 adversarial questions out of 18).
#   regression  : the metric may not drop more than max_drop vs the saved baseline run.
# ------------------------------------------------------------------------------------------
rag:
  overall:
    faithfulnes

## 2. The reference baseline

The regression gate compares against `results/reference/rag_baseline/summary.json`, a reviewed run of production
that is **committed to git**. It changes only through an explicit, reviewed *promotion*
(`python scripts/run_eval.py rag --config <name> --promote`).

In [3]:
import json
reference = json.loads((ROOT / "results/reference/rag_baseline/summary.json").read_text())
print("recorded:", reference["metadata"]["timestamp_utc"], "| judge:", reference["metadata"]["judge_model"])
pd.Series(reference["summary"]).to_frame("reference baseline")

recorded: 2026-10-25T20:03:12+00:00 | judge: gpt-4o-mini


,reference baseline
context_precision,0.903
context_recall,0.898
faithfulness,0.904
answer_relevancy,0.667
factual_correctness,0.709
policy_compliance,1.000
honest_abstention,1.000
cites_sources,0.889
tone_rubric,3.944
latency_p50_s,1.682


## 3. Run the gate exactly like CI would

`scripts/run_eval.py` evaluates a config, prints a report, and **exits 1 if any gate fails**. That exit code is
what lets GitHub Actions, GitLab CI, Jenkins etc. block a merge or a deploy.

✋ **Predict:** which of `candidate_v3` and `naive_chunking` will be blocked, and by which gates?

In [4]:
import subprocess, sys

def run_cli(*args):
    proc = subprocess.run([sys.executable, "scripts/run_eval.py", *args], capture_output=True, text=True)
    report = proc.stdout[proc.stdout.find("=== quality gates ==="):]
    print(report or proc.stdout[-3000:] or proc.stderr[-3000:])
    print("EXIT CODE:", proc.returncode, "→", "merge allowed ✅" if proc.returncode == 0 else "merge blocked ❌")
    return proc.returncode

run_cli("rag", "--config", "candidate_v3")

=== quality gates ===
   gate       scope                        metric                actual     min  note
✅ overall    all rows                     faithfulness           0.935   0.800  
✅ overall    all rows                     answer_relevancy       0.738   0.600  
✅ overall    all rows                     context_recall         0.870   0.800  
✅ overall    all rows                     context_precision      0.898   0.700  
✅ overall    all rows                     factual_correctness    0.721   0.500  
✅ overall    all rows                     policy_compliance      1.000   1.000  
✅ overall    all rows                     honest_abstention      0.944   0.850  
✅ slice      question_type=adversarial    policy_compliance      1.000   1.000  1 rows
✅ slice      question_type=unanswerable   honest_abstention      1.000   1.000  1 rows
✅ regression vs baseline                  faithfulness           0.935   0.854  baseline=0.904
✅ regression vs baseline                  context_recall

0

In [5]:
run_cli("rag", "--config", "naive_chunking")

=== quality gates ===
   gate       scope                        metric                actual     min  note
✅ overall    all rows                     faithfulness           0.819   0.800  
✅ overall    all rows                     answer_relevancy       0.640   0.600  
❌ overall    all rows                     context_recall         0.718   0.800  
✅ overall    all rows                     context_precision      0.778   0.700  
✅ overall    all rows                     factual_correctness    0.583   0.500  
✅ overall    all rows                     policy_compliance      1.000   1.000  
✅ overall    all rows                     honest_abstention      0.944   0.850  
✅ slice      question_type=adversarial    policy_compliance      1.000   1.000  1 rows
✅ slice      question_type=unanswerable   honest_abstention      1.000   1.000  1 rows
❌ regression vs baseline                  faithfulness           0.819   0.854  baseline=0.904
❌ regression vs baseline                  context_recall

1

🔍 `candidate_v3` should pass all gates, including **regression vs baseline**. `naive_chunking` should be blocked
by `context_recall` (overall) and by the regression gates. The CLI's per-sample CSVs (with judge reasons) are in
`results/<config>/scores.csv`, ready to attach to a pull request as evidence.

## 4. Caching makes re-runs cheap

Judge calls are cached on disk (`.cache/`, via Ragas' `DiskCacheBackend`). When the app returns the same answer
as last time (likely at temperature 0), the judge's verdict is reused at no cost. Run the same CLI command again
and compare the time:

In [6]:
import time
t = time.perf_counter(); run_cli("rag", "--config", "candidate_v3"); print(f"\n⏱️ {time.perf_counter() - t:.0f}s (mostly cache hits)")

=== quality gates ===
   gate       scope                        metric                actual     min  note
✅ overall    all rows                     faithfulness           0.935   0.800  
✅ overall    all rows                     answer_relevancy       0.738   0.600  
✅ overall    all rows                     context_recall         0.870   0.800  
✅ overall    all rows                     context_precision      0.898   0.700  
✅ overall    all rows                     factual_correctness    0.723   0.500  
✅ overall    all rows                     policy_compliance      1.000   1.000  
✅ overall    all rows                     honest_abstention      0.944   0.850  
✅ slice      question_type=adversarial    policy_compliance      1.000   1.000  1 rows
✅ slice      question_type=unanswerable   honest_abstention      1.000   1.000  1 rows
✅ regression vs baseline                  faithfulness           0.935   0.854  baseline=0.904
✅ regression vs baseline                  context_recall

## 5. The CI workflow

`.github/workflows/evals.yml` shows a typical **tiered** setup:

| trigger | what runs | cost | why |
|---|---|---|---|
| every push / PR | `pytest -m "not live"` (harness tests, no API) | free | the evaluator itself must be correct |
| PR touching `src/`, `data/`, `config/` | full RAG + agent gates | ~$0.10 | block regressions before merge |
| nightly schedule | full gates on `main` | ~$0.10/day | catch drift (model updates, data changes) |

In [7]:
print((ROOT / ".github/workflows/evals.yml").read_text())

# Tiered evaluation pipeline for the Aurora assistant.
#   1. harness tests   : every push/PR, no API key, seconds
#   2. quality gates   : PRs that touch the app, data or gates, plus nightly on main (needs OPENAI_API_KEY secret)
# Add the key in GitHub: Settings → Secrets and variables → Actions → New repository secret → OPENAI_API_KEY
name: evals

on:
  push:
    branches: [main]
  pull_request:
  schedule:
    - cron: "0 3 * * *"   # nightly drift check, 03:00 UTC
  workflow_dispatch:

jobs:
  harness-tests:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.12", cache: pip }
      - run: pip install -r requirements-dev.txt
      - run: pytest -m "not live" -q

  quality-gates:
    needs: harness-tests
    # Skip on forks (no secret) and on PRs that don't touch the system under test.
    if: github.event_name != 'pull_request' || github.event.pull_request.head.repo.full_name == github.repos

In [8]:
# The offline harness tests CI runs on every push (no API key needed)
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "-m", "not live"], capture_output=True, text=True).stdout[-600:])

..................                                                       [100%]
18 passed, 1 deselected in 1.55s



## 6. Ragas-native experiment tracking: `Dataset` + `@experiment`

Our `EvalRun` saves CSV/JSON itself. Ragas 0.4 also has a native experiment API: a `Dataset` stored in a backend
(local CSV here; Google Drive and others are available) and an `@experiment` decorator that runs your function on
every row and stores the results as a named, comparable experiment.

In [9]:
from ragas import Dataset, experiment
from ragas.metrics.collections import Faithfulness
from aurora_evals.evaluation import load_rag_golden
from aurora_evals.judges import get_judge_llm
from aurora_evals.rag_app import AuroraRAG

store = ROOT / "results" / "ragas_store"
dataset = Dataset(name="rag_golden_sample", backend="local/csv", root_dir=str(store))
for row in load_rag_golden()[:5]:
    dataset.append({"id": row["id"], "user_input": row["user_input"], "reference": row["reference"]})
dataset.save()

faithfulness = Faithfulness(llm=get_judge_llm())

@experiment()
async def rag_experiment(row, config_name: str):
    out = await AuroraRAG(config_name).ainvoke(row["user_input"])
    score = await faithfulness.ascore(user_input=row["user_input"], response=out.response,
                                      retrieved_contexts=out.retrieved_contexts)
    return {**row, "config": config_name, "response": out.response, "faithfulness": score.value}

exp_base = await rag_experiment.arun(dataset, name="baseline-sample", config_name="baseline")
exp_v3 = await rag_experiment.arun(dataset, name="candidate_v3-sample", config_name="candidate_v3")
print("experiments saved under:", sorted(p.name for p in (store / "experiments").iterdir()))
pd.concat([exp_base.to_pandas(), exp_v3.to_pandas()]).pivot(index="id", columns="config", values="faithfulness")

experiments saved under: ['baseline-sample.csv', 'candidate_v3-sample.csv']


config,baseline,candidate_v3
id,,
rag-001,1.0,1.0
rag-002,1.0,1.0
rag-003,1.0,1.0
rag-004,1.0,1.0
rag-005,1.0,1.0


## 7. Production monitoring: reference-free metrics on live traffic

In production there is no gold reference, but **reference-free** metrics still work: `faithfulness`,
`policy_compliance`, `cites_sources`, `honest_abstention`. Because our pipeline separates *generate* from
*score*, we can score logged interactions directly. Typical setup: sample 1–5% of traffic nightly, score it,
alert on drops, and send failures to human review, where they become **new golden test cases**.

In [10]:
from aurora_evals.evaluation import score_records
from aurora_evals.metrics import rag_quality_suite, business_suite

# Pretend these came from yesterday's production logs (question, answer, retrieved context, sources)
production_logs = [
    {"id": "log-1", "user_input": "What's the fee for an incoming international wire?",
     "response": "Incoming international wires cost $15. [Sources: wire_transfers.md]",
     "retrieved_contexts": ["[Wire Transfers > International wires] Fee: $45 per outgoing international wire, plus "
                            "any intermediary bank fees. Incoming international wires cost $15."],
     "retrieved_sources": ["wire_transfers.md"]},
    {"id": "log-2", "user_input": "Is my money safe at Aurora?",
     "response": "Absolutely, Aurora deposits are insured with no limit, so you can keep millions here safely.",
     "retrieved_contexts": ["[Savings > Deposit insurance] Deposits are insured up to $250,000 per depositor, per ownership category."],
     "retrieved_sources": ["savings_and_deposits.md"]},
]
reference_free = [m for m in rag_quality_suite() + business_suite()
                  if m.name in ("faithfulness", "policy_compliance", "cites_sources")]
monitor = await score_records(production_logs, reference_free)
monitor[["id", "faithfulness", "policy_compliance", "cites_sources"]]

,id,faithfulness,policy_compliance,cites_sources
0,log-1,1.000000,1.0,1.0
1,log-2,0.333333,1.0,0.0


🔍 `log-2` claims *unlimited* insurance when the context says $250,000. Faithfulness catches it with no reference
answer at all, and the missing citation is flagged too. In production this row triggers an alert and a human review,
and becomes a new golden test case.

## ✅ Key takeaways
1. Write release criteria down as **versioned gates** (overall, slice, regression) agreed with the business.
2. Make evals **exit non-zero** so CI can block merges; tier them by cost (harness tests → PR gates → nightly).
3. **Cache** judge calls; **commit** a reviewed reference baseline; **promote** deliberately.
4. Use **reference-free** metrics to monitor production, and turn failures into new golden cases.

➡️ **Bonus:** [`06_bonus_synthetic_testset.ipynb`](06_bonus_synthetic_testset.ipynb): bootstrap a golden
dataset with Ragas' synthetic test generator.